# ROS09 · rosbag2 - record, inspect, analyze, replay

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS09_rosbag2_record_inspect_analyze_replay.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 50 min

**Skill Hub concepts:** rosbag2 recording and analysis

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS09")

## Map: a bag is a time-ordered log of serialized messages

```
 live topics ──ros2 bag record──> run1/  (metadata.yaml + run1_0.mcap: topic, type, QoS, receive time, CDR bytes)
                                    │
            ros2 bag info run1 ─────┤──── rosbag2_py.SequentialReader ──> deserialize_message ──> numpy / pandas
                                    │
 your node <──ros2 bag play run1 --rate 2.0 --clock──┘     (replays with the original timing, or faster)
```

| Command | Use |
|---|---|
| `ros2 bag record -o run1 /a /b` (or `-a` for all) | capture; stop with Ctrl+C (SIGINT) so the file is closed cleanly |
| `ros2 bag info run1` | duration, message counts, types, storage |
| `ros2 bag play run1 --rate 2.0 --loop` | replay faster, forever |
| `ros2 bag play run1 --clock` + node param `use_sim_time:=true` | nodes follow the bag's time, so timers and TF lookups match the recording |

Why it matters: record the inputs of a failure once, then replay them into your node as many times as you need to
debug or as a regression test, with no robot.

In [ ]:
from roboprep import ros

In [ ]:
%%writefile temp_node.py
import math
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from std_msgs.msg import Float32


class Temp(Node):
    def __init__(self):
        super().__init__("joint1_sensor")
        self.pub = self.create_publisher(Float32, "/joint1/temperature", 10)
        self.k = 0
        self.create_timer(0.05, self.tick)

    def tick(self):
        self.pub.publish(Float32(data=40.0 + 5.0 * math.sin(0.05 * self.k)))
        self.k += 1


def main():
    rclpy.init()
    try:
        rclpy.spin(Temp())
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

## Task 1: record a few seconds
Start the node, start `ros2 bag record` into folder `run1` for `/joint1/temperature` only, wait 5 s (the recorder
needs a moment to discover the topic), stop everything. The grader wants at least 40 messages.

In [ ]:
import shutil, time
shutil.rmtree("run1", ignore_errors=True)
ros.start("python temp_node.py")
time.sleep(1.5)
# TODO: ros.start("ros2 bag record ...", log="record.log")
time.sleep(5.0)
ros.stop_all()
time.sleep(1)
ros.ros("ros2 bag info run1")

In [ ]:
lab.check("record")

## Task 2: analyze the bag
`analyze_bag.py BAG` prints one JSON line: `{"count": ..., "rate_hz": ..., "mean": ..., "min": ..., "max": ...}`
for `/joint1/temperature`, where `rate_hz = (count - 1) / (last_time - first_time)` from the bag's timestamps.

In [ ]:
%%writefile analyze_bag.py
import json
import sys
import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

reader = rosbag2_py.SequentialReader()
reader.open(rosbag2_py.StorageOptions(uri=sys.argv[1], storage_id=""), rosbag2_py.ConverterOptions("", ""))
types = {t.name: t.type for t in reader.get_all_topics_and_types()}
# TODO: loop with reader.has_next() / reader.read_next(), deserialize, collect, print one JSON line

In [ ]:
ros.ros("python analyze_bag.py run1")
lab.check("analyze")

In [ ]:
%%writefile analyze_bag.cpp
// C++ (rclcpp) version of Task 2 (optional, same checker). Grade it with lab.check("analyze", "analyze_bag.cpp"):
// the grader builds it with colcon as package rdl_analyze_bag, executable analyze_bag (depends on rclcpp, rosbag2_cpp, std_msgs).
#include <cstdio>
#include <string>
#include <vector>
#include <algorithm>
#include <rclcpp/serialization.hpp>
#include <rosbag2_cpp/reader.hpp>
#include <std_msgs/msg/float32.hpp>

int main(int argc, char **argv) {
    if (argc < 2) { std::fprintf(stderr, "usage: analyze_bag <bag folder>\n"); return 1; }
    rosbag2_cpp::Reader reader;
    reader.open(argv[1]);
    // TODO: while (reader.has_next()): read_next(), keep /joint1/temperature, deserialize with
    //       rclcpp::Serialization<std_msgs::msg::Float32>, collect recv_timestamp and data,
    //       then print one JSON line with count, rate_hz, mean, min, max
}

In [ ]:
# lab.check("analyze", "analyze_bag.cpp")  # uncomment after writing the C++ cell above

Replay at double speed and watch the rate change:

In [ ]:
ros.start("ros2 bag play run1 --rate 2.0")
ros.ros("timeout 2 ros2 topic hz /joint1/temperature", timeout=10)
ros.stop_all()

## Checkpoint: Math
**M1 (Apply).** A 20 Hz topic recorded for 3 s. About how many messages?

**M2 (Apply).** A 3 s bag played with `--rate 2.0` takes how many seconds?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Apply).** Your perception node crashes once an hour on the robot. What do you record to debug it at your desk?
- a) the node's output topics
- b) the node's input topics (and `/tf`, `/tf_static`), so you can replay exactly what it saw into a fresh copy of the node
- c) only `/rosout`

**K2 (Understand).** What do `--clock` and `use_sim_time` do during playback, and what breaks without them?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros09.h`).
3. In `roboprep/checks/ros09.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()